# Lead Scoring: Catching Data Leakage Before It Inflates Your Metrics

Most notebooks on this dataset jump straight to modeling. This one starts with
a leakage audit — because two columns in this dataset (`Tags`, `Lead Quality`)
will silently give you a 95%+ accuracy model that has learned nothing useful.

**What this notebook covers:**
1. A leakage audit with evidence (not guesswork) — the part most public notebooks on this dataset skip
2. Fair model comparison (Logistic Regression → Random Forest → XGBoost) with consistent imbalance handling
3. 5-fold CV instead of a single train/val split — every number below is mean ± std, not a lucky split
4. Hyperparameter tuning and a business-relevant KPI (conversion lift), not just accuracy

**Full pipeline + a deployed Streamlit app (single-lead and batch CSV scoring)
with SHAP explanations:** [github.com/akash25aiml-coder/lead-scoring-project](https://github.com/akash25aiml-coder/lead-scoring-project)

*Akash Upadhyay — [LinkedIn](https://linkedin.com/in/akash-a-upadhyay) · [GitHub](https://github.com/akash25aiml-coder)*


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, recall_score, ConfusionMatrixDisplay
import shap

RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

## 2. Load data

In [ ]:
import glob
path = glob.glob('/kaggle/input/**/Leads.csv', recursive=True)[0]
df = pd.read_csv(path)
print('Shape:', df.shape)
df.head()

## 3. Quick EDA

Class balance and missingness — the two things that decide every downstream choice.

In [ ]:
print('Target balance:')
print(df['Converted'].value_counts(normalize=True).round(3))

miss = (df.isnull().mean() * 100).round(1)
print('\nColumns with missing values:')
print(miss[miss > 0].sort_values(ascending=False))

## 4. Leakage audit — read this before you trust any model on this dataset

Three columns look like useful features but are actually recorded **after**
the outcome is already known, or only assigned to leads a sales rep has
already engaged with. Training on them gives an inflated, useless model.

In [ ]:
# Tags: "current status of the lead" per the data dictionary — a sales
# rep's post-hoc note, not a pre-conversion signal
top_tags = df['Tags'].value_counts().head(6).index
print("Tags vs Converted (top tags):")
print(pd.crosstab(df[df['Tags'].isin(top_tags)]['Tags'],
                   df[df['Tags'].isin(top_tags)]['Converted'],
                   normalize='index').round(2))

In [ ]:
# Lead Quality / Lead Profile: only assigned once a rep has already
# engaged with the lead — missingness itself leaks the outcome
for col in ['Lead Quality', 'Lead Profile']:
    tmp = df.copy()
    tmp['_missing'] = tmp[col].isnull()
    print(f"\n{col} — conversion rate by missingness:")
    print(tmp.groupby('_missing')['Converted'].mean().round(3))

**Verdict: drop `Tags`, `Lead Quality`, `Lead Profile`.**

Sanity-checked one more suspicious value — `Last Activity == "Converted to Lead"`
sounds like leakage by name. It converts at **12.6%**, well *below* the 38.5%
baseline — a CRM lifecycle label, not an outcome marker. Confirmed safe, kept in.

In [ ]:
LEAKAGE_COLS = ['Tags', 'Lead Quality', 'Lead Profile']

ZERO_VARIANCE_COLS = [
    'Magazine', 'Receive More Updates About Our Courses',
    'Update me on Supply Chain Content', 'Get updates on DM Content',
    'I agree to pay the amount through cheque', 'Search', 'Newspaper Article',
    'X Education Forums', 'Newspaper', 'Digital Advertisement',
    'Through Recommendations', 'Do Not Call',
]  # 100% or >99.8% single value — statistically inert

ID_COLS = ['Prospect ID', 'Lead Number']
SPARSE_DROP_COLS = ['How did you hear about X Education']  # 78.5% missing once "Select" -> NaN

df_clean = df.replace('Select', pd.NA)
df_clean = df_clean.drop(columns=LEAKAGE_COLS + ZERO_VARIANCE_COLS + SPARSE_DROP_COLS, errors='ignore')
print('Shape after cleaning:', df_clean.shape, ' — dropped', df.shape[1] - df_clean.shape[1], 'columns')

## 5. Feature prep

Rare categories (<1% frequency) in high-cardinality columns get grouped into
`"Other"` — fit on the training split only, applied to val/test — otherwise
tree models overfit on categories with 1-2 samples.

In [ ]:
y = df_clean['Converted']
X = df_clean.drop(columns=['Converted'] + ID_COLS, errors='ignore')
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

HIGH_CARD_COLS = ['Country', 'Specialization', 'Last Activity', 'Last Notable Activity', 'City']

def fit_rare_map(df_tr, cols, threshold=0.01):
    m = {}
    for c in cols:
        freq = df_tr[c].value_counts(normalize=True, dropna=True)
        m[c] = set(freq[freq < threshold].index)
    return m

def group_rare(df_, freq_map, cols):
    df_ = df_.copy()
    for c in cols:
        if c in freq_map:
            df_[c] = df_[c].apply(lambda x: 'Other' if x in freq_map[c] else x)
    return df_

freq_map = fit_rare_map(X_train, HIGH_CARD_COLS)
X_train = group_rare(X_train, freq_map, HIGH_CARD_COLS)
X_val = group_rare(X_val, freq_map, HIGH_CARD_COLS)
X_test = group_rare(X_test, freq_map, HIGH_CARD_COLS)

In [ ]:
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Not Provided')),
                       ('encode', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

## 6. Fair model comparison (5-fold CV, consistent imbalance handling)

Same `class_weight`/`scale_pos_weight` logic across all three models — otherwise
the comparison is apples-to-oranges.

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'recall': 'recall', 'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'}

results = []
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        'model': name,
        'recall': f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        'pr_auc': f"{scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}",
        'roc_auc': f"{scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}",
    })
pd.DataFrame(results)

**Interesting result before tuning:** Logistic Regression's recall is
competitive with — sometimes better than — Random Forest at default
hyperparameters. Trees don't automatically win. Tuning is what decides it.

## 7. Hyperparameter tuning (RandomizedSearchCV, scoring = PR-AUC)

In [ ]:
rf_dist = {'clf__n_estimators': [200, 300, 500], 'clf__max_depth': [8, 12, 16, None],
           'clf__min_samples_leaf': [1, 2, 5, 10], 'clf__max_features': ['sqrt', 'log2']}
xgb_dist = {'clf__n_estimators': [200, 300, 500], 'clf__max_depth': [3, 4, 6, 8],
            'clf__learning_rate': [0.01, 0.05, 0.1, 0.2], 'clf__subsample': [0.7, 0.85, 1.0],
            'clf__colsample_bytree': [0.7, 0.85, 1.0]}

tuned = {}
for name, clf, dist in [('Random Forest', models['Random Forest'], rf_dist), ('XGBoost', models['XGBoost'], xgb_dist)]:
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    search = RandomizedSearchCV(pipe, dist, n_iter=20, scoring='average_precision', cv=3, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    print(f'{name} best CV PR-AUC: {search.best_score_:.3f}')

final_model = tuned['XGBoost']  # winner after tuning on recall, PR-AUC, and ROC-AUC

## 8. Final model — test set, touched exactly once

In [ ]:
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, test_pred, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, test_proba), 3))
print('PR-AUC :', round(average_precision_score(y_test, test_proba), 3))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=['Not Converted', 'Converted'], cmap='Blues', ax=ax)
plt.title('XGBoost (tuned) — Test Set')
plt.show()

## 9. Business KPI: does this actually help prioritize leads?

In [ ]:
order = np.argsort(test_proba)[::-1]
top_decile_n = int(0.1 * len(test_proba))
top_decile_conv = y_test.values[order[:top_decile_n]].mean()
baseline_conv = y_test.mean()
print(f'Top-decile conversion rate: {top_decile_conv:.3f}')
print(f'Baseline conversion rate  : {baseline_conv:.3f}')
print(f'Lift: {top_decile_conv/baseline_conv:.2f}x — a sales team working only the top 10% scored leads sees this multiple of the baseline conversion rate')

## 10. What's actually driving predictions (SHAP)

In [ ]:
prep = final_model.named_steps['prep']
clf = final_model.named_steps['clf']
X_sample = X_train.sample(500, random_state=RANDOM_STATE)
X_t = prep.transform(X_sample)
feat_names = prep.get_feature_names_out()

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_t)
shap.summary_plot(shap_values, X_t, feature_names=feat_names, max_display=12)

## Takeaways

- Three columns (`Tags`, `Lead Quality`, `Lead Profile`) looked predictive and
  weren't — they were recording the outcome, not predicting it. Checking this
  is the difference between a real model and an inflated one.
- The simplest model (Logistic Regression) held its own against untuned trees.
  Tuning, not model complexity, is what won it for XGBoost.
- Top-decile scored leads convert at a meaningfully higher rate than baseline
  — this is the number that would actually matter to a sales team.

**Full repo (tests, CI, Streamlit app with batch CSV scoring + live SHAP
explanations):** [github.com/akash25aiml-coder/lead-scoring-project](https://github.com/akash25aiml-coder/lead-scoring-project)

If this caught something useful, an upvote helps it reach more people looking
at this dataset. Feedback welcome.
